In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

必要となるPythonライブラリをインポートする。 

In [ ]:
# データ操作と数値計算
import pandas as pd
import numpy as np
import re # 正規表現用

# 自然言語処理 (NLP) 関連
import nltk
from nltk.corpus import stopwords # ストップワード除去用
from nltk.tokenize import word_tokenize # 単語分割用

# Hugging Face Transformers と PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW # AdamWはtorch.optimから直接インポート
from transformers import get_linear_schedule_with_warmup

# モデル評価とデータ分割
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# 可視化ライブラリ
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# その他のユーティリティ
import os # ファイルパス操作用

データセットのパスを定義する。

In [ ]:
train_path = '/kaggle/input/nlp-getting-started/train.csv'
test_path = '/kaggle/input/nlp-getting-started/test.csv'
sample_submission_path = '/kaggle/input/nlp-getting-started/sample_submission.csv'

Kaggleの入力ディレクトリから
訓練データ (train.csv)
テストデータ (test.csv)
提出サンプルファイル (sample_submission.csv) を読み込む。 

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

読み込んだ各データの形状と最初の数行を表示して確認する。
欠損値の情報も表示する。

In [ ]:
print("\n訓練データ (train_df) の情報:")
print(train_df.shape)
print(train_df.head())

print("\nテストデータ (test_df) の情報:")
print(test_df.shape)
print(test_df.head())

print("\n訓練データの欠損値:")
print(train_df.isnull().sum())
print("\nテストデータの欠損値:")
print(test_df.isnull().sum())

欠損値を「no_keyword」や「no_location」という文字列で埋める。
これらは直接的な特徴量としては利用しないが、データの一貫性を保つために処理を施す。

In [ ]:
train_df['keyword'] = train_df['keyword'].fillna('no_keyword')
test_df['keyword'] = test_df['keyword'].fillna('no_keyword')
train_df['location'] = train_df['location'].fillna('no_location')
test_df['location'] = test_df['location'].fillna('no_location')

ツイート本文に対して、
URL (https://..., www....) の除去、
HTMLタグ (<...> ) の除去、
絵文字の除去を行う。
これらの要素はBERTの学習においてノイズとなる可能性があり、ツイートの本来の意味を歪める可能性があるため除去する。

In [ ]:
def clean_text_for_bert(text):

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'<.*?>', '', text)

    emoji_pattern = re.compile(
        '['
        '\U0001F600-\U0001F64F' '\U0001F300-\U0001F5FF'
        '\U0001F680-\U0001F6FF' '\U0001F1E0-\U0001F1FF'
        '\U00002702-\U000027B0' '\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    return text

train_df['cleaned_text_bert'] = train_df['text'].apply(clean_text_for_bert)
test_df['cleaned_text_bert'] = test_df['text'].apply(clean_text_for_bert)

訓練データとテストデータのツイート数を表示する。
また、先ほどの処理ができているか確認するため、最初の訓練ツイートの例を表示する。

In [ ]:
sentences = train_df['cleaned_text_bert'].values
labels = train_df['target'].values
test_sentences = test_df['cleaned_text_bert'].values

print(f"\n訓練データのツイート数: {len(sentences)}")
print(f"テストデータのツイート数: {len(test_sentences)}")
print(f"最初の訓練ツイートの例: {sentences[0]}")

データから初期の洞察を得て、立てた仮説を視覚的に検証するためのグラフを生成する。
まず、災害ツイート（target=1）と非災害ツイート（target=0）それぞれにおいて、欠損値を除く上位20個のキーワードの出現頻度を棒グラフで比較する。
これにより、特定のキーワードが災害ツイートに偏って出現する傾向にあるかを確認する。 

In [ ]:
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['keyword'] != 'no_keyword'], y='keyword', hue='target',
              order=train_df[train_df['keyword'] != 'no_keyword']['keyword'].value_counts().index[:20])
plt.title('Keyword Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Keyword')
plt.tight_layout()
plt.show()

次に、災害ツイートと非災害ツイートそれぞれについてワードクラウドを作成する。
これにより、両者の間で頻出する単語の違いを視覚的に捉えられる。 

In [ ]:
stop_words = set(stopwords.words('english'))

def generate_wordcloud(text_data, title):

    all_words = ' '.join(text_data).lower()
    words = word_tokenize(all_words)
    filtered_words = [word for word in words if word.isalnum() and word not in stop_words]
    cleaned_text = ' '.join(filtered_words)

    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          collocations=False).generate(cleaned_text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

generate_wordcloud(train_df[train_df['target'] == 1]['cleaned_text_bert'], 'Word Cloud for Disaster Tweets')

generate_wordcloud(train_df[train_df['target'] == 0]['cleaned_text_bert'], 'Word Cloud for Non-Disaster Tweets')

次に、災害ツイートと非災害ツイートそれぞれにおいて、欠損値を除く上位20個の地域の頻度を棒グラフで比較する。
これにより、特定の地域からのツイートが災害関連である傾向にあるかを確認する。 

In [ ]:
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['location'] != 'no_location'], y='location', hue='target',
              order=train_df[train_df['location'] != 'no_location']['location'].value_counts().index[:20])
plt.title('Location Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Location')
plt.tight_layout()
plt.show()

最後にキーワードと地域それぞれの欠損値の有無を示すグラフを作成する。
欠損値の有無がターゲット（災害/非災害）とどのように関連しているかを棒グラフで可視化する。
これらのグラフデータから、立てた仮説を視覚的に検証できる。

In [ ]:
train_df['keyword_missing'] = train_df['keyword'] == 'no_keyword'
train_df['location_missing'] = train_df['location'] == 'no_location'

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(data=train_df, x='keyword_missing', hue='target')
plt.title('Relationship between Keyword Missingness and Target')
plt.xlabel('Keyword Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.subplot(1, 2, 2)
sns.countplot(data=train_df, x='location_missing', hue='target')
plt.title('Relationship between Location Missingness and Target')
plt.xlabel('Location Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.tight_layout()
plt.show()

BERTトークナイザーのロードとテキストのエンコードを行う。
Hugging FaceのBertTokenizer.from_pretrained('bert-base-uncased')を用いて、BERTモデルが理解できる形式（トークンID、アテンションマスクなど）にテキストを変換する。 
encode_texts関数は、各ツイートをBERTの入力形式に変換するが、これには特殊トークンの追加（文頭の[CLS]、文末の[SEP]）、最大長 (MAX_LEN=128) へのパディングまたは切り捨て、アテンションマスクの生成、そしてPyTorchテンソルへの変換が含まれる。 
訓練データとテストデータに対してこのエンコード処理を適用し、BERTモデルへの入力準備を整える。 

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

def encode_texts(tokenizer, texts, max_len=128):
    input_ids = []
    attention_masks = []
    for text in texts:
        encoded_dict = tokenizer.encode_plus(
                            text,
                            add_special_tokens = True,
                            max_length = max_len,
                            padding = 'max_length',
                            truncation = True,
                            return_attention_mask = True,
                            return_tensors = 'pt',
                       )

        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
    
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)

    return input_ids, attention_masks

# 訓練データとテストデータをBERT形式に変換
MAX_LEN = 128
train_inputs, train_masks = encode_texts(tokenizer, sentences, MAX_LEN)
test_inputs, test_masks = encode_texts(tokenizer, test_sentences, MAX_LEN)

# ラベルをPyTorchテンソルに変換
train_labels = torch.tensor(labels)
print(f"エンコードされた訓練データの入力ID形状: {train_inputs.shape}")
print(f"エンコードされた訓練データのアテンションマスク形状: {train_masks.shape}")
print(f"訓練ラベルの形状: {train_labels.shape}")

モデルの汎化性能を評価するために、訓練データセットを訓練用と検証用に分割する（ここでは10%を検証データとして使用）。

In [ ]:
train_inputs, validation_inputs, train_masks, validation_masks, train_labels, validation_labels = train_test_split(
    train_inputs, train_masks, train_labels, random_state=42, test_size=0.1
)

print(f"分割後の訓練データセットの形状: {train_inputs.shape}")
print(f"分割後の検証データセットの形状: {validation_inputs.shape}")

効率的なバッチ処理とモデル学習のために、PyTorchのTensorDatasetとDataLoaderを作成する。 
訓練データはランダムにシャッフルし、検証データとテストデータは順次処理する。 

In [ ]:
batch_size = 32
# 訓練データ用
train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# 検証データ用
validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels)
validation_sampler = SequentialSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

# テストデータ用 (ラベルがないため入力とマスクのみ)
test_data = TensorDataset(test_inputs, test_masks)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

BERTモデルをロードし、学習のための設定を行い、実際にモデルをファインチューニングする。 
まず、GPUが利用可能かどうかを確認し、利用可能であればGPUを、そうでなければCPUを計算デバイスとして設定する。
BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)を用いて、事前学習済みのBERTモデルをロードする。
num_labels=2は、災害/非災害の2クラス分類を行うことを示す。
モデルを適切なデバイス（GPUまたはCPU）に移動する。

In [ ]:
print("\n--- BERTモデルの準備 ---")
# GPUが利用可能か確認し、デバイスを設定する
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPUが利用可能です:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('GPUは利用できません。CPUを使用します。')

# BERTモデルをロード (シーケンス分類用)
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,      # 出力クラス数 (0:非災害, 1:災害)
    output_attentions = False,
    output_hidden_states = False,
)

# モデルをGPUに移動 (GPUが利用可能な場合)
model.to(device)

オプティマイザとスケジューラを設定する。
モデルのパラメータを更新するためのオプティマイザとして、BERTのファインチューニングに推奨されるAdamWを使用し、学習率（lr）やイプシロン（eps）を設定する。 
学習率を動的に調整するためのget_linear_schedule_with_warmupスケジューラを設定する。
これにより、学習の初期段階で学習率を徐々に上げ、その後線形に減少させることで、安定した学習を促進する。
モデルの予測結果（ロジット）と実際のラベルを比較し、正解率（Accuracy）を計算するための補助関数であるflat_accuracy関数を定義する。

In [ ]:
# オプティマイザの設定
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

epochs = 3

# 学習率スケジューラの設定 (ウォームアップ付き線形スケジューラ)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# 評価関数 (Accuracy)
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return accuracy_score(labels_flat, pred_flat)

指定されたエポック数（ここでは3エポック）だけ学習を繰り返す。 
訓練フェーズと検証フェーズに分かれて学習する。

訓練フェーズ:
モデルを訓練モード（model.train()）に設定する。 
訓練データローダーからバッチごとにデータを取得し、デバイスに移動する。 
勾配をゼロクリアし、順伝播（モデルにデータを入力して出力とロスを計算）を実行する。 
逆伝播（ロスに基づいて勾配を計算）を行い、勾配クリッピング（勾配爆発を防ぐため）を適用する。 
オプティマイザとスケジューラを使ってモデルのパラメータと学習率を更新する。 
定期的に訓練ロスを表示し、学習の進捗を監視する。

検証フェーズ: 
各エポックの終わりに、モデルを評価モード（model.eval()）に設定する。 
検証データローダーからデータを取得し、勾配計算を無効にした状態でモデルにデータを入力する。 
モデルの出力（ロジット）と実際のラベルを用いて、正解率とF1スコアを計算し、表示する。
これにより、モデルが訓練データだけでなく、未知のデータに対してもどれだけ汎化できているかを評価する。

In [ ]:
print("\n--- BERTモデルのファインチューニングを開始します ---")
# 訓練フェーズ
for epoch_i in range(0, epochs):
    print(f"\n======== エポック {epoch_i + 1} / {epochs} ========")
    print("訓練中...")

    model.train()

    total_loss = 0

    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask,
                        labels=b_labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        if step % 200 == 0 and not step == 0:
            print(f"  Batch {step:>5,}  Loss: {loss.item():.2f}")
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"  平均訓練ロス: {avg_train_loss:.2f}")
    
# 検証フェーズ
    print("\n検証中...")
    model.eval()

    total_eval_accuracy = 0
    total_eval_f1 = 0
    nb_eval_steps = 0

    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        with torch.no_grad():
            outputs = model(b_input_ids,
                            token_type_ids=None,
                            attention_mask=b_input_mask,
                            labels=b_labels)

        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        total_eval_accuracy += flat_accuracy(logits, label_ids)
        total_eval_f1 += f1_score(label_ids, np.argmax(logits, axis=1).flatten())

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_f1 = total_eval_f1 / len(validation_dataloader)
    print(f"  検証Accuracy: {avg_val_accuracy:.2f}")
    print(f"  検証F1スコア: {avg_val_f1:.2f}")

print("\n--- ファインチューニングが完了しました ---")

テストデータでの予測を行う。
モデルを評価モードに設定したまま、テストデータローダーからデータを取得する。 
勾配計算を無効にした状態でモデルにテストデータを入力し、予測結果（ロジット）を取得する。 
これらのロジットをNumpy配列に変換し、最も高い確率を持つクラス（0または1）を最終的な予測結果として選択する。

In [ ]:
print("\n--- テストデータでの予測を開始します ---")
model.eval()

predictions = []

for batch in test_dataloader:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)
    logits = outputs.logits
    logits = logits.detach().cpu().numpy()
    predictions.append(logits)

final_predictions_bert = np.argmax(np.concatenate(predictions, axis=0), axis=1)

print(f"最終予測結果の形状: {final_predictions_bert.shape}")
print(f"最終予測結果の最初の5件: {final_predictions_bert[:5]}")

Kaggleの提出形式であるsample_submission.csvをコピーし、そのtargetカラムに最終予測結果を代入する。作成された提出用DataFrameの最初の数行を表示して内容を確認します。 
最後に、このDataFrameをsubmission.csvという名前でCSVファイルとして保存する。

In [ ]:
# 提出用DataFrameを作成
submission_df_bert = sample_submission_df.copy()
submission_df_bert['target'] = final_predictions_bert

print("\n生成された提出ファイルのプレビュー:")
print(submission_df_bert.head())

# 提出ファイルをCSV形式で保存
submission_df_bert.to_csv('submission.csv', index=False)

このノートブックは、データの前処理からモデルの学習、評価、そして最終的な提出ファイルの生成まで、一連のデータ分析パイプラインを網羅しており、災害ツイート分類という課題に対するBERTモデルの有効性を示している。 